# 🔍 GPU-Accelerated AML Network Analysis
### Detecting money laundering rings in transaction networks — CPU vs GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adibis-git/fraud-detection-gpu-demo/blob/main/aml_gpu_network_analysis.ipynb)

---

**Choose your context — this demo is relevant if you are:**

| Your organisation | Your AML pain point | What this demo shows |
|---|---|---|
| 🏦 Retail / commercial bank | RBI PMLA compliance, STR filing backlog | How GPU cuts investigation time per alert |
| 💳 NBFC / lending platform | Fraud ring detection across borrower networks | How GPU finds connected bad actors in real time |
| ⚡ Payment network / fintech | Transaction monitoring at millions of tx/hour | How GPU keeps pace without queue build-up |
| 🛡️ Insurance company | Claims fraud ring detection at renewal scale | How GPU processes the full network, not a sample |

**The core question this notebook answers:**
> *Your AML model works. But at peak load — month-end settlement, festive season, IPO day — does your infrastructure keep up? Or does the queue grow faster than you can clear it?*

**Runtime:** ~12 minutes on Colab free tier (T4 GPU)  
**Prerequisite:** Runtime → Change runtime type → **T4 GPU** ← do this first

---

## 1. Environment Setup

We use NVIDIA RAPIDS cuGraph — a GPU-accelerated graph analytics library.  
The key point: **your existing NetworkX code runs on GPU with a single environment variable.**  
No re-skilling. No rewrite. Zero code change.

In [ ]:
import subprocess, sys

print("Installing RAPIDS cuGraph (GPU graph analytics)...")
print("This takes ~3 minutes on first run — grab a coffee.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--extra-index-url", "https://pypi.nvidia.com",
     "cugraph-cu12", "nx-cugraph-cu12", "cudf-cu12"],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "networkx", "matplotlib", "pandas", "numpy", "scipy"],
    check=True
)
print("\n✅ Setup complete")

In [ ]:
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"✅ GPU ready: {result.stdout.strip()}")
else:
    print("⚠️  No GPU detected. Runtime → Change runtime type → T4 GPU → Reconnect")

## 2. Build the Transaction Network

We generate a synthetic AML transaction network that mirrors real BFSI characteristics:
- **500,000 account nodes** (customers, merchants, intermediaries)
- **2,000,000 transaction edges** (fund flows between accounts)
- **Embedded laundering rings** — tightly connected sub-graphs where funds cycle before exiting
- **Mule accounts** — high-throughput nodes that bridge rings to clean accounts

> This scale represents approximately 6 hours of transaction volume for a mid-size Indian private bank.

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import time

np.random.seed(42)

N_ACCOUNTS    = 500_000
N_LEGIT_TX    = 1_800_000
N_RINGS       = 150          # money laundering rings
RING_SIZE     = 8            # accounts per ring
N_MULES       = 500          # bridge accounts

print("Building transaction network...")

# Legitimate transactions — power-law degree distribution (realistic)
sources = np.random.zipf(1.5, N_LEGIT_TX) % N_ACCOUNTS
targets = np.random.randint(0, N_ACCOUNTS, N_LEGIT_TX)
amounts = np.random.lognormal(mean=9, sigma=2, size=N_LEGIT_TX)  # log-normal amounts

edges = pd.DataFrame({
    'source': sources,
    'target': targets,
    'amount': amounts.astype(np.float32),
    'is_laundering': 0
})
edges = edges[edges['source'] != edges['target']]  # remove self-loops

# Embed money laundering rings (layering phase: funds cycle within ring)
ring_edges = []
ring_accounts = set()
for i in range(N_RINGS):
    ring_nodes = list(range(
        N_ACCOUNTS + i * RING_SIZE,
        N_ACCOUNTS + (i + 1) * RING_SIZE
    ))
    ring_accounts.update(ring_nodes)
    for j in range(RING_SIZE):
        ring_edges.append({
            'source': ring_nodes[j],
            'target': ring_nodes[(j + 1) % RING_SIZE],
            'amount': float(np.random.uniform(50_000, 500_000)),
            'is_laundering': 1
        })
    # Connect ring to mule account (integration phase)
    mule = np.random.randint(0, N_MULES)
    ring_edges.append({
        'source': ring_nodes[0],
        'target': mule,
        'amount': float(np.random.uniform(100_000, 1_000_000)),
        'is_laundering': 1
    })

ring_df = pd.DataFrame(ring_edges)
edges = pd.concat([edges, ring_df], ignore_index=True)

total_nodes = N_ACCOUNTS + N_RINGS * RING_SIZE

print(f"\nNetwork built:")
print(f"  Nodes (accounts)      : {total_nodes:>10,}")
print(f"  Edges (transactions)  : {len(edges):>10,}")
print(f"  Laundering rings      : {N_RINGS:>10,}")
print(f"  Suspicious edges      : {edges['is_laundering'].sum():>10,} ({edges['is_laundering'].mean()*100:.1f}%)")
print(f"  Total transaction vol : ₹{edges['amount'].sum()/1e7:>8.1f} Cr")

## 3. What the AML Algorithms Do

We run two standard AML graph techniques:

| Algorithm | What it detects | AML application |
|---|---|---|
| **PageRank** | High-influence accounts that many funds flow through | Identifies money mules and structuring hubs |
| **Louvain Community Detection** | Tightly connected clusters of accounts | Detects layering rings — accounts cycling funds between each other |

Both are in active use in production AML systems at large banks.

## 4. ⚡ Benchmark: CPU vs GPU

### 4a. PageRank — money mule identification

**The zero-code-change story:**  
Below you will see the exact same NetworkX `pagerank()` call run on CPU, then on GPU.  
The only difference is one environment variable. Your team does not need to learn a new library.

In [ ]:
print("Building NetworkX graph from edge list...")
t0 = time.perf_counter()
G = nx.from_pandas_edgelist(
    edges,
    source='source',
    target='target',
    edge_attr='amount',
    create_using=nx.DiGraph()
)
build_time = time.perf_counter() - t0
print(f"✅ Graph built in {build_time:.1f}s  |  {G.number_of_nodes():,} nodes  |  {G.number_of_edges():,} edges")

In [ ]:
import os

# ── CPU PageRank ──────────────────────────────────────────────────────────────
print("Running PageRank on CPU (NetworkX)...")
os.environ.pop('NX_CUGRAPH_AUTOCONFIG', None)  # ensure GPU backend is OFF

N_RUNS = 3
cpu_pr_times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    pr_cpu = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-5)
    cpu_pr_times.append(time.perf_counter() - t0)

cpu_pr_time = min(cpu_pr_times)  # best of N runs
print(f"  CPU PageRank : {cpu_pr_time:.2f}s")

In [ ]:
# ── GPU PageRank — zero code change ──────────────────────────────────────────
print("Running PageRank on GPU (same NetworkX call, cuGraph backend)...")
os.environ['NX_CUGRAPH_AUTOCONFIG'] = 'True'  # one line: enable GPU backend

# Warm-up
_ = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-5)

gpu_pr_times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    pr_gpu = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-5)
    gpu_pr_times.append(time.perf_counter() - t0)

gpu_pr_time = min(gpu_pr_times)
pr_speedup = cpu_pr_time / gpu_pr_time

print(f"  GPU PageRank : {gpu_pr_time:.2f}s")
print(f"\n{'='*45}")
print(f"  PageRank speedup : {pr_speedup:.1f}×  faster on GPU")
print(f"{'='*45}")
print(f"\nCode change required: 1 environment variable. That's it.")

### 4b. Louvain Community Detection — transaction ring discovery

Louvain finds tightly connected clusters. In AML, each cluster is a potential layering ring —  
a group of accounts cycling funds between each other to obscure the trail.

In [ ]:
# Convert to undirected for community detection (standard practice)
G_undirected = G.to_undirected()

# ── CPU Louvain (NetworkX greedy modularity as CPU baseline) ──────────────────
print("Running community detection on CPU (NetworkX greedy modularity)...")
os.environ.pop('NX_CUGRAPH_AUTOCONFIG', None)

t0 = time.perf_counter()
communities_cpu = list(nx.community.greedy_modularity_communities(G_undirected))
cpu_louvain_time = time.perf_counter() - t0

print(f"  CPU : {cpu_louvain_time:.2f}s  |  {len(communities_cpu):,} communities found")

In [ ]:
import cudf
import cugraph

# ── GPU Louvain (native cuGraph — true Louvain algorithm) ─────────────────────
print("Running Louvain community detection on GPU (cuGraph)...")

edges_gpu = cudf.DataFrame({
    'src': edges['source'].values,
    'dst': edges['target'].values,
    'weight': edges['amount'].values
})

G_cu = cugraph.from_cudf_edgelist(
    edges_gpu, source='src', destination='dst',
    edge_attr='weight', create_using=cugraph.Graph()
)

# Warm-up
_, _ = cugraph.louvain(G_cu, max_level=10, resolution=1.0)

gpu_louvain_times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    parts_gpu, modularity_gpu = cugraph.louvain(G_cu, max_level=10, resolution=1.0)
    gpu_louvain_times.append(time.perf_counter() - t0)

gpu_louvain_time = min(gpu_louvain_times)
n_communities_gpu = parts_gpu['partition'].nunique()
louvain_speedup = cpu_louvain_time / gpu_louvain_time

print(f"  GPU : {gpu_louvain_time:.2f}s  |  {n_communities_gpu:,} communities found")
print(f"  Modularity score : {modularity_gpu:.4f}")
print(f"\n{'='*45}")
print(f"  Louvain speedup  : {louvain_speedup:.1f}×  faster on GPU")
print(f"{'='*45}")

## 5. Identify Suspicious Accounts

In [ ]:
# Top PageRank accounts = highest-priority mule candidates
pr_series = pd.Series(pr_gpu).sort_values(ascending=False)
top_mule_candidates = pr_series.head(20)

# Flag small tight communities as laundering ring candidates
community_map = parts_gpu.to_pandas().set_index('vertex')['partition']
community_sizes = community_map.value_counts()
ring_candidates = community_sizes[
    (community_sizes >= 5) & (community_sizes <= 15)
]

print(f"AML alert summary")
print("-" * 40)
print(f"Top mule candidates (high PageRank) : {len(top_mule_candidates):,} accounts")
print(f"Potential ring communities (5–15 sz): {len(ring_candidates):,} clusters")
print(f"Accounts in ring clusters            : {ring_candidates.sum():,}")
print()
print("Top 5 mule candidates by PageRank score:")
for acc, score in top_mule_candidates.head(5).items():
    print(f"  Account {acc:>7d}  →  PageRank: {score:.6f}")

## 6. Visualise — Transaction Ring vs Normal Network

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig = plt.figure(figsize=(15, 6))

# ── Left: benchmark bar chart ─────────────────────────────────────────────────
ax1 = fig.add_subplot(1, 2, 1)
algorithms  = ['PageRank\n(mule detection)', 'Louvain\n(ring detection)']
cpu_times   = [cpu_pr_time, cpu_louvain_time]
gpu_times   = [gpu_pr_time, gpu_louvain_time]
speedups    = [pr_speedup, louvain_speedup]

x = range(len(algorithms))
w = 0.35
bars_cpu = ax1.bar([i - w/2 for i in x], cpu_times, w,
                   label='CPU', color='#E24B4A', alpha=0.9)
bars_gpu = ax1.bar([i + w/2 for i in x], gpu_times, w,
                   label='GPU', color='#639922', alpha=0.9)

ax1.set_xticks(list(x))
ax1.set_xticklabels(algorithms, fontsize=10)
ax1.set_ylabel('Time (seconds)')
ax1.set_title('CPU vs GPU — AML Algorithm Runtime', fontweight='bold')
ax1.legend()
ax1.spines[['top', 'right']].set_visible(False)

for i, (bc, bg, sp) in enumerate(zip(bars_cpu, bars_gpu, speedups)):
    ax1.text(bc.get_x() + bc.get_width()/2, bc.get_height() + 0.3,
             f'{cpu_times[i]:.1f}s', ha='center', va='bottom', fontsize=9)
    ax1.text(bg.get_x() + bg.get_width()/2, bg.get_height() + 0.3,
             f'{gpu_times[i]:.1f}s', ha='center', va='bottom', fontsize=9)
    mid_x = (bc.get_x() + bc.get_width() + bg.get_x()) / 2
    mid_y = max(cpu_times[i], gpu_times[i]) * 0.6
    ax1.text(mid_x, mid_y, f'{sp:.0f}×\nfaster',
             ha='center', va='center', fontsize=13,
             fontweight='bold', color='#3B6D11',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#EAF3DE', alpha=0.8))

# ── Right: sample transaction ring visualisation ──────────────────────────────
ax2 = fig.add_subplot(1, 2, 2)

# Show a sample ring subgraph
sample_ring_community = ring_candidates.index[0]
ring_nodes_sample = community_map[
    community_map == sample_ring_community
].index.tolist()[:12]

# Add a few normal accounts connected to the ring
normal_accounts = list(range(10, 25))
subgraph_nodes  = set(ring_nodes_sample + normal_accounts[:8])
H = G.subgraph(subgraph_nodes).copy()

pos = nx.spring_layout(H, seed=42, k=2)
node_colors = [
    '#E24B4A' if n in ring_nodes_sample else '#B5D4F4'
    for n in H.nodes()
]
node_sizes = [
    500 if n in ring_nodes_sample else 200
    for n in H.nodes()
]

nx.draw_networkx(
    H, pos, ax=ax2,
    node_color=node_colors,
    node_size=node_sizes,
    edge_color='#888780',
    arrows=True,
    with_labels=False,
    width=1.2,
    alpha=0.85
)

ring_patch   = mpatches.Patch(color='#E24B4A', label='Flagged ring accounts')
normal_patch = mpatches.Patch(color='#B5D4F4', label='Normal accounts')
ax2.legend(handles=[ring_patch, normal_patch], loc='lower right', fontsize=9)
ax2.set_title('Sample detected laundering ring\n(Louvain community detection)', fontweight='bold')
ax2.axis('off')

plt.suptitle(
    f'GPU-Accelerated AML · {total_nodes:,} accounts · {len(edges):,} transactions',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('aml_gpu_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as aml_gpu_benchmark.png")

## 7. The Business Case — Cost per AML Investigation

In [ ]:
# AWS on-demand pricing (us-east-1, 2025)
cpu_price_hr = 0.096   # c5.2xlarge  (8 vCPU, 16GB)
gpu_price_hr = 0.526   # g4dn.xlarge (4 vCPU, 16GB, T4 GPU)

# Full network scan time
total_cpu_time = cpu_pr_time + cpu_louvain_time
total_gpu_time = gpu_pr_time + gpu_louvain_time

# Scans per hour
cpu_scans_hr = 3600 / total_cpu_time
gpu_scans_hr = 3600 / total_gpu_time

# Cost per full network scan
cost_cpu_per_scan = (total_cpu_time / 3600) * cpu_price_hr
cost_gpu_per_scan = (total_gpu_time / 3600) * gpu_price_hr

# At 24/7 operation
daily_cpu_scans = cpu_scans_hr * 24
daily_gpu_scans = gpu_scans_hr * 24

print("Business impact summary")
print("=" * 55)
print(f"Network size          : {total_nodes:,} accounts, {len(edges):,} transactions")
print()
print(f"{'Metric':<35} {'CPU':>10} {'GPU':>10}")
print("-" * 55)
print(f"{'Full network scan time (s)':<35} {total_cpu_time:>10.1f} {total_gpu_time:>10.1f}")
print(f"{'Scans per hour':<35} {cpu_scans_hr:>10.0f} {gpu_scans_hr:>10.0f}")
print(f"{'Cost per scan (USD)':<35} ${cost_cpu_per_scan:>9.4f} ${cost_gpu_per_scan:>9.4f}")
print(f"{'Scans per day (24/7)':<35} {daily_cpu_scans:>10.0f} {daily_gpu_scans:>10.0f}")
print("-" * 55)

overall_speedup = total_cpu_time / total_gpu_time
cost_saving_pct = (1 - cost_gpu_per_scan / cost_cpu_per_scan) * 100
print(f"\nGPU is {overall_speedup:.0f}× faster end-to-end")
print(f"Cost per investigation reduced by {cost_saving_pct:.0f}%")
print()
print("Regulatory context (India):")
print("  RBI requires STR filing within 7 days of suspicion.")
print(f"  On CPU: {total_cpu_time:.0f}s per full network scan means you can run")
print(f"  {daily_cpu_scans:.0f} scans/day — real-time detection is not feasible.")
print(f"  On GPU: {total_gpu_time:.1f}s per scan — continuous monitoring becomes possible.")

---

## What this means for your AML stack

| Scenario | CPU infrastructure | GPU infrastructure |
|---|---|---|
| Daily transaction network scan | Batch job, results next morning | Continuous, near real-time |
| STR filing window (RBI: 7 days) | Manual prioritisation, risk of breach | Automated ring flagging within hours |
| Peak volume (festive, month-end) | Queue grows, investigations delayed | Absorbs load within SLA |
| Investigator workload | High false-positive rate (broad net) | Tighter communities = higher precision |
| Regulatory audit trail | Sampling only — full scan too slow | Full network evidence, every cycle |

**The biggest misconception:** GPU infrastructure requires a specialised team.  
**The reality:** As shown above, your existing NetworkX-based models run on GPU with a single environment variable. No rewrite. No new hire.

---

### Ready to see what this looks like on your transaction data?

This benchmark runs on synthetic data. A production deployment layers on:
- Your core banking system as the transaction source
- Entity resolution across accounts, UBO chains, and beneficial ownership
- FATF typology rules as pre-filters before graph analytics
- Explainability layer for investigator handoff and regulatory documentation

**If you are running batch AML on CPU today, let's do a 20-minute architecture review of what a GPU-accelerated pipeline would look like for your volumes.**

---
*Notebook 2 of a series · GPU-Accelerated BFSI ML · Built with NVIDIA RAPIDS cuGraph + NetworkX*  
*[Notebook 1: GPU-Accelerated Fraud Detection (XGBoost, PaySim)](https://colab.research.google.com/github/adibis-git/fraud-detection-gpu-demo/blob/main/fraud_detection_gpu_demo.ipynb)*